In [30]:
import evaluate
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
import numpy as np
import torch

# Load pre-trained T5-small (original, not fine-tuned)
model_name = "t5-small"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

# Load a small subset of CNN/DailyMail validation set
dataset = load_dataset("cnn_dailymail", "3.0.0", split="validation[:20]")

# Preprocess input/output pairs
inputs = ["summarize: " + article[:100] for article in dataset["article"]]
references = dataset["highlights"]

# Tokenize inputs
inputs_tokenized = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Generate summaries using the original model
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs_tokenized["input_ids"],
        attention_mask=inputs_tokenized["attention_mask"],
        max_length=128
    )

# Decode model outputs and compute ROUGE
predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)

# Print ROUGE-L score
print("\n🔍 ROUGE scores before fine-tuning:")
print(f"ROUGE-1: {round(scores['rouge1'] * 100, 2)}")
print(f"ROUGE-2: {round(scores['rouge2'] * 100, 2)}")
print(f"ROUGE-L: {round(scores['rougeL'] * 100, 2)} ✅")


🔍 ROUGE scores before fine-tuning:
ROUGE-1: 16.81
ROUGE-2: 5.37
ROUGE-L: 13.4 ✅


In [29]:
from urllib.parse import quote
import requests
import evaluate
from datasets import load_dataset
import json
from urllib.parse import unquote

# Step 1: Load validation data
dataset = load_dataset("cnn_dailymail", "3.0.0", split="validation[:20]")

# Step 2: Prepare inputs and references
inputs = ["summarize: " + article[:100] for article in dataset["article"]]  
references = dataset["highlights"]
predictions = []

ENDPOINT = "https://r2fo8rz7f0.execute-api.us-east-1.amazonaws.com/V1"

print("[INFO] Querying endpoint...")
for text in inputs:
    try:
        safe_query = quote(text, safe="")
        response = requests.get(ENDPOINT, params={"query": safe_query})
        if response.status_code == 200:
            outer = response.json()
            inner = json.loads(outer["body"])
            summary = inner.get("summary", "")
            decoded_summary = unquote(summary) 
            predictions.append(decoded_summary.strip())
            # print(decoded_summary.strip())
        else:
            print(f"[WARN] Failed request: {response.status_code}")
            predictions.append("")
    except Exception as e:
        print(f"[ERROR] {e}")
        predictions.append("")

# Step 4: Compute ROUGE
rouge = evaluate.load("rouge")
scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)

# Step 5: Print final scores
print("\n✅ ROUGE Scores from Fine-Tuned Model Endpoint:")
print(f"ROUGE-1: {round(scores['rouge1'] * 100, 2)}")
print(f"ROUGE-2: {round(scores['rouge2'] * 100, 2)}")
print(f"ROUGE-L: {round(scores['rougeL'] * 100, 2)}")

[INFO] Querying endpoint...

✅ ROUGE Scores from Fine-Tuned Model Endpoint:
ROUGE-1: 18.11
ROUGE-2: 5.43
ROUGE-L: 15.35
